In [1]:
!pip install firebase-admin


[notice] A new release of pip is available: 23.0.1 -> 26.0
[notice] To update, run: pip install --upgrade pip


In [4]:
import json
import re
import firebase_admin
from firebase_admin import credentials, firestore

# 1. Inicialização
SERVICE_ACCOUNT = "/Users/leosaracino/Documents/Faculdade/TCC-MOTIVA-APP/scripts_gerais/motiva-8b82f-firebase-adminsdk-fbsvc-14d8d2b5e8.json"
try:
    firebase_admin.get_app()
    print("Firebase já inicializado.")
except ValueError:
    cred = credentials.Certificate(SERVICE_ACCOUNT)
    firebase_admin.initialize_app(cred)
    print("Firebase inicializado.")

db = firestore.client()

# 2. Carrega e normaliza o JSON
with open('exercises.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
    print("JSON carregado.")
if isinstance(data, list):
    exercises = data
    print("JSON é uma lista de exercícios.")
elif isinstance(data, dict) and 'movimentos' in data and isinstance(data['movimentos'], dict):
    exercises = list(data['movimentos'].values())
    print("JSON é um dict com chave 'movimentos'.")
elif isinstance(data, dict):
    exercises = list(data.values())
    print("JSON é um dict de exercícios.")
else:
    raise ValueError("Formato JSON não suportado. Deve ser array de objetos ou dict de dicts.")

# 3. Gerador de IDs (snake_case) caso falte
def slugify(name: str) -> str:
    s = name.strip().lower()
    s = re.sub(r'[^a-z0-9]+', '_', s)
    return s.strip('_')


# 4. Importação com verificação de existência
print("--- Iniciando Loop ---")

for ex in exercises:
    # garante displayName
    disp = ex.get('displayName')
    
    if not disp:
        print("⚠️  Exercício sem displayName, pulando:", ex)
        continue

    # ID do doc
    doc_id = ex.get('name') or slugify(disp)
    ex['name'] = doc_id
    
    print(f"\n🔍 Processando: {doc_id}")

    doc_ref = db.collection('movimentos').document(doc_id)
    
    # VERIFICAÇÃO COM TIMEOUT
    try:
        print("   ...Contatando Firestore...")
        # Adicionei timeout=10 para não ficar travado para sempre
        doc_snap = doc_ref.get(timeout=10)
        
        if doc_snap.exists:
            print(f"   ⏭️  Já existe no banco, pulando.")
            continue
            
    except Exception as e:
        print(f"   ❌ ERRO DE CONEXÃO ao buscar {doc_id}. O script vai parar.")
        print(f"   Detalhe do erro: {e}")
        break # Para o script para você não esperar a toa

    print(f"   Documento novo. Gravando...")
    
    # GRAVAÇÃO
    try:
        doc_ref.set(ex)
        print(f"   ✔️  Sucesso: {doc_id} gravado.")
    except Exception as e:
        print(f"   ❌ Falha ao gravar {doc_id}: {e}")

print("✨ Fim do processo!")

Firebase já inicializado.
JSON carregado.
JSON é uma lista de exercícios.
--- Iniciando Loop ---

🔍 Processando: air_squat
   ...Contatando Firestore...


KeyboardInterrupt: 